In [1]:
# =============================================================================
# Combine Original + Augmented Data for RoBERTa Fine-Tuning
# Shuffle and Split: 70% train / 15% val / 15% test
# =============================================================================

import pandas as pd
from sklearn.model_selection import train_test_split

# ── Update these paths to your files ────────────────────────────────────────
ORIGINAL_REAL  = r"../aug/real_120k_matched.csv"          # original real (text, label)
ORIGINAL_FAKE  = r"../aug/fake_120k_matched.csv"          # original fake (text, label)
AUGMENTED_REAL = r"../aug/real_120k_t5paws_aug.csv"       # augmented real (text, label, augtype)
AUGMENTED_FAKE = r"../aug/fake_120k_t5paws_aug.csv"       # augmented fake (text, label, augtype)

OUTPUT_FOLDER = r"../Data_aug"                              # where to save train/val/test CSVs

In [3]:
# ── Load all data ───────────────────────────────────────────────────────────
# ...existing code...

# ── Load all data ───────────────────────────────────────────────────────────
print("Loading original and augmented CSVs...")

df_orig_real = pd.read_csv(ORIGINAL_REAL, usecols=["text", "label"])
df_orig_fake = pd.read_csv(ORIGINAL_FAKE, usecols=["text", "label"])

df_aug_real = pd.read_csv(AUGMENTED_REAL, usecols=["text", "label", "augtype"])
df_aug_fake = pd.read_csv(AUGMENTED_FAKE, usecols=["text", "label", "augtype"])

df_aug_real = df_aug_real[df_aug_real["augtype"] != "fallback"]
df_aug_fake = df_aug_fake[df_aug_fake["augtype"] != "fallback"]

# Drop 'augtype' column after filtering, as originals don't have it
df_aug_real = df_aug_real.drop(columns=["augtype"])
df_aug_fake = df_aug_fake.drop(columns=["augtype"])

# ...existing code...

Loading original and augmented CSVs...


In [4]:
# ── Combine all into one big DF ─────────────────────────────────────────────
df_all = pd.concat([df_orig_real, df_orig_fake, df_aug_real, df_aug_fake], ignore_index=True)

# Basic cleaning (drop NaN text, ensure label 0/1)
df_all = df_all.dropna(subset=["text"]).reset_index(drop=True)
df_all["label"] = df_all["label"].astype(int)

print(f"Total combined rows: {len(df_all):,}")
print("Label balance:")
print(df_all["label"].value_counts(normalize=True).round(3) * 100)

Total combined rows: 479,892
Label balance:
label
0    50.0
1    50.0
Name: proportion, dtype: float64


In [5]:
# ── Shuffle well ────────────────────────────────────────────────────────────
df_shuffled = df_all.sample(frac=1, random_state=42).reset_index(drop=True)  # seed for reproducibility

# ── Split: 70% train / 15% val / 15% test ──────────────────────────────────
# Stratify by label to keep real/fake balance in each split

train_df, temp_df = train_test_split(
    df_shuffled,
    test_size=0.30,             # 30% for val + test
    stratify=df_shuffled["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,             # split remaining 30% into 15/15
    stratify=temp_df["label"],
    random_state=42
)

In [7]:
# ── Save to CSVs ────────────────────────────────────────────────────────────
from pathlib import Path


train_df.to_csv(Path(OUTPUT_FOLDER) / "train.csv", index=False)
val_df.to_csv(Path(OUTPUT_FOLDER) / "val.csv", index=False)
test_df.to_csv(Path(OUTPUT_FOLDER) / "test.csv", index=False)

print("\nSplits saved:")
print(f"Train: {len(train_df):,} rows (~70%)")
print(f"Val:   {len(val_df):,} rows (~15%)")
print(f"Test:  {len(test_df):,} rows (~15%)")

print("\nLabel balance check (train):")
print(train_df["label"].value_counts(normalize=True).round(3) * 100)

print("\nAll done. Files saved to:", OUTPUT_FOLDER)


Splits saved:
Train: 335,924 rows (~70%)
Val:   71,984 rows (~15%)
Test:  71,984 rows (~15%)

Label balance check (train):
label
0    50.0
1    50.0
Name: proportion, dtype: float64

All done. Files saved to: ../Data_aug


In [8]:
# =============================================================================
# Load & Inspect Train / Val / Test CSVs (after combining original + augmented)
# Prints info only — no changes made
# =============================================================================

import pandas as pd
from pathlib import Path

# ── Update these paths if your files are saved somewhere else ───────────────
FOLDER = r"../Data_aug"   # or r"C:\gyanendra\aug\data\" or wherever you saved them

TRAIN_PATH = Path(FOLDER) / "train.csv"
VAL_PATH   = Path(FOLDER) / "val.csv"
TEST_PATH  = Path(FOLDER) / "test.csv"

# ── Load all three splits ───────────────────────────────────────────────────
print("Loading datasets...\n")

try:
    train_df = pd.read_csv(TRAIN_PATH)
    val_df   = pd.read_csv(VAL_PATH)
    test_df  = pd.read_csv(TEST_PATH)
except FileNotFoundError as e:
    print(f"Error: One or more files not found → {e}")
    print("Check your paths and make sure the CSVs were created successfully.")
    raise

# ── Basic info for each split ───────────────────────────────────────────────
def print_split_info(name, df):
    print(f"{'='*60}")
    print(f" {name.upper()} SPLIT")
    print(f"{'='*60}")
    print(f"Shape (rows × columns): {df.shape}")
    print(f"Total rows: {len(df):,}")
    print("\nColumns:", list(df.columns))
    
    if "text" in df.columns and "label" in df.columns:
        print("\nLabel distribution:")
        label_dist = df["label"].value_counts(normalize=True).round(4) * 100
        print(label_dist)
        print("\nRaw label counts:")
        print(df["label"].value_counts())
    
    print("\nFirst 3 rows preview:")
    display(df.head(3))
    
    if "text" in df.columns:
        avg_len = df["text"].str.len().mean()
        avg_words = df["text"].str.split().str.len().mean()
        print(f"\nAverage text length (chars): {avg_len:.0f}")
        print(f"Average words per text: {avg_words:.1f}")
    
    print("\n")

print_split_info("Train", train_df)
print_split_info("Validation", val_df)
print_split_info("Test", test_df)

# ── Overall summary ─────────────────────────────────────────────────────────
total_rows = len(train_df) + len(val_df) + len(test_df)
print(f"{'='*60}")
print(f" OVERALL SUMMARY")
print(f"{'='*60}")
print(f"Total examples across all splits: {total_rows:,}")
print(f"Train ratio: {len(train_df)/total_rows*100:.1f}%")
print(f"Val ratio:   {len(val_df)/total_rows*100:.1f}%")
print(f"Test ratio:  {len(test_df)/total_rows*100:.1f}%")

print("\nAll splits loaded and inspected successfully.")

Loading datasets...

 TRAIN SPLIT
Shape (rows × columns): (335924, 2)
Total rows: 335,924

Columns: ['text', 'label']

Label distribution:
label
0    50.01
1    49.99
Name: proportion, dtype: float64

Raw label counts:
label
0    167984
1    167940
Name: count, dtype: int64

First 3 rows preview:


,text,label
0,Corrections An article on Oct. 26 about a golf...,0
1,"Get Back BEATLE boots, as cobbler lore has it,...",0
2,How to Make the Most of Free Hotel Amenities P...,0



Average text length (chars): 823
Average words per text: 135.6


 VALIDATION SPLIT
Shape (rows × columns): (71984, 2)
Total rows: 71,984

Columns: ['text', 'label']

Label distribution:
label
0    50.01
1    49.99
Name: proportion, dtype: float64

Raw label counts:
label
0    35997
1    35987
Name: count, dtype: int64

First 3 rows preview:


,text,label
0,Warning! We Told You Something is Coming and N...,1
1,Salmonella Outbreak at Tarheel Q Hits 70 Salmo...,1
2,Medical Marijuana Send to Email Address Your N...,1



Average text length (chars): 824
Average words per text: 135.8


 TEST SPLIT
Shape (rows × columns): (71984, 2)
Total rows: 71,984

Columns: ['text', 'label']

Label distribution:
label
0    50.01
1    49.99
Name: proportion, dtype: float64

Raw label counts:
label
0    35997
1    35987
Name: count, dtype: int64

First 3 rows preview:


,text,label
0,The total financial collapse and public enslav...,1
1,The “Maleficent” actress is reportedly fightin...,0
2,Best Flashlight 2014 Best Flashlight 2014 % of...,1



Average text length (chars): 822
Average words per text: 135.4


 OVERALL SUMMARY
Total examples across all splits: 479,892
Train ratio: 70.0%
Val ratio:   15.0%
Test ratio:  15.0%

All splits loaded and inspected successfully.
